In [8]:
import gymnasium as gym
import numpy as np
import random
from collections import deque
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

In [9]:
# Create the environment
env = gym.make("CartPole-v1")

# Get the number of states and actions
n_states = env.observation_space.shape[0]
n_actions = env.action_space.n

print(f"Number of states (Inputs): {n_states}")
print(f"Number of actions (Outputs): {n_actions}")

Number of states (Inputs): 4
Number of actions (Outputs): 2


In [10]:
# Create a visualization environment
env_viz = gym.make("CartPole-v1", render_mode="human")
state, info = env_viz.reset()

print("Starting random run... (Watch the popup window)")

for _ in range(100): # Run for 100 frames
    env_viz.render()
    
    # Pick a random action (0 or 1)
    random_action = env_viz.action_space.sample()
    
    # Take the action
    next_state, reward, terminated, truncated, info = env_viz.step(random_action)
    
    # If the pole fell (terminated) or time ran out (truncated), reset
    if terminated or truncated:
        state, info = env_viz.reset()

env_viz.close()
print("Random run finished.")

Starting random run... (Watch the popup window)
Random run finished.


In [11]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        
        # Hyperparameters
        self.memory = deque(maxlen=2000) # Replay memory
        self.gamma = 0.95    # Discount rate
        self.epsilon = 1.0   # Exploration rate
        self.epsilon_min = 0.01 
        self.epsilon_decay = 0.995 
        self.learning_rate = 0.001
        
        # The Brain
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential()
        model.add(Input(shape=(self.state_size,)))
        model.add(Dense(24, activation='relu'))
        model.add(Dense(24, activation='relu'))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        # Ensure state is (1, 4) before predicting
        state = np.reshape(state, [1, self.state_size]) 
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)
        
        states = np.array([i[0] for i in minibatch])
        states = np.squeeze(states) 
        
        next_states = np.array([i[3] for i in minibatch])
        next_states = np.squeeze(next_states)
        # -----------------------

        # Now the shapes match what Keras expects
        qs = self.model.predict(states, verbose=0)
        qs_next = self.model.predict(next_states, verbose=0)
        
        X = []
        y = []

        for i in range(batch_size):
            state, action, reward, next_state, done = minibatch[i]
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(qs_next[i])
            
            target_f = qs[i]
            target_f[action] = target
            
            X.append(states[i]) # Use the squeezed state
            y.append(target_f)
            
        self.model.fit(np.array(X), np.array(y), epochs=1, verbose=0)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [12]:
# Initialize Agent
agent = DQNAgent(n_states, n_actions)
batch_size = 32

# CHANGE THIS: Increase episodes to give it time to learn
episodes = 100 

print(f"Starting training for {episodes} episodes...")

for e in range(episodes):
    state, info = env.reset()
    state = np.reshape(state, [1, n_states]) # Reshape for Keras
    score = 0
    done = False
    
    while not done:
        # 1. Decide action
        action = agent.act(state)
        
        # 2. Take action
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state = np.reshape(next_state, [1, n_states]) # Reshape for Keras
        
        # Reward shaping: punish strictly if it falls
        # If done (failed), give -10 reward. Else give normal reward (+1)
        reward = reward if not done else -10
        
        # 3. Remember
        agent.remember(state, action, reward, next_state, done)
        
        # 4. Advance state
        state = next_state
        score += 1
        
        # 5. Learn from past
        agent.replay(batch_size)
        
        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {score}, Epsilon: {agent.epsilon:.2f}")
            if score > 450:
                print(f"\n SOLVED! Stopping early at Episode {e+1}")
                agent.model.save("cartpole_solved.keras") # Save the brain
                break
            break
        
    if score > 450:
        break
            
print("Training Complete!")

Starting training for 100 episodes...
Episode: 1/100, Score: 30, Epsilon: 1.00
Episode: 2/100, Score: 16, Epsilon: 0.93
Episode: 3/100, Score: 70, Epsilon: 0.65
Episode: 4/100, Score: 31, Epsilon: 0.56
Episode: 5/100, Score: 40, Epsilon: 0.46
Episode: 6/100, Score: 24, Epsilon: 0.41
Episode: 7/100, Score: 28, Epsilon: 0.35
Episode: 8/100, Score: 12, Epsilon: 0.33
Episode: 9/100, Score: 8, Epsilon: 0.32
Episode: 10/100, Score: 11, Epsilon: 0.30
Episode: 11/100, Score: 8, Epsilon: 0.29
Episode: 12/100, Score: 35, Epsilon: 0.24
Episode: 13/100, Score: 30, Epsilon: 0.21
Episode: 14/100, Score: 35, Epsilon: 0.18
Episode: 15/100, Score: 45, Epsilon: 0.14
Episode: 16/100, Score: 27, Epsilon: 0.12
Episode: 17/100, Score: 24, Epsilon: 0.11
Episode: 18/100, Score: 25, Epsilon: 0.10
Episode: 19/100, Score: 26, Epsilon: 0.08
Episode: 20/100, Score: 21, Epsilon: 0.08
Episode: 21/100, Score: 70, Epsilon: 0.05
Episode: 22/100, Score: 46, Epsilon: 0.04
Episode: 23/100, Score: 31, Epsilon: 0.04
Episode

In [14]:
# Visualization mode
env_viz = gym.make("CartPole-v1", render_mode="human")
state, info = env_viz.reset()

print("Starting Trained Agent... (Watch the popup window)")
score = 0
done = False

while not done:
    env_viz.render()
    
    # We force the model to predict (no random moves)
    state_reshaped = np.reshape(state, [1, n_states])
    action = np.argmax(agent.model.predict(state_reshaped, verbose=0)[0])
    
    next_state, reward, terminated, truncated, info = env_viz.step(action)
    done = terminated or truncated
    state = next_state
    score += 1

print(f"Final Score: {score}")
env_viz.close()

Starting Trained Agent... (Watch the popup window)
Final Score: 294
